### NSFW Detection Dataset - EDA and preprocessing

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image, ImageOps
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from torchvision import models, transforms
import torch

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "data" else NOTEBOOK_DIR
DATA_ROOT = PROJECT_ROOT / "data"
RAW_ROOT = DATA_ROOT / "raw"
INTERIM_ROOT = DATA_ROOT / "interim"
PROCESSED_ROOT = DATA_ROOT / "processed"
for folder in (RAW_ROOT, INTERIM_ROOT, PROCESSED_ROOT):
    folder.mkdir(parents=True, exist_ok=True)

DATASET_NAME = 'NSFW Detection Dataset (HuggingFace)'
DEFAULT_SOURCE = '../data/raw/nsfw-detection-dataset/nsfw_detection_dataset.parquet'
POSITIVE_ALIASES = {'nsfw', 'unsafe', 'porn', 'sexy', 'adult', 'explicit', '1', 'true'}

sns.set_theme(style="whitegrid")


In [ ]:
def _read_tabular(path: Path) -> pd.DataFrame:
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    if path.suffix in {".csv", ".tsv"}:
        sep = "	" if path.suffix == ".tsv" else ","
        return pd.read_csv(path, sep=sep)
    raise ValueError(f"Unsupported tabular file: {path}")


def _resolve_image(value, source_root: Path) -> Path | None:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    if isinstance(value, Path):
        return value
    if isinstance(value, str):
        path = Path(value)
        return path if path.is_absolute() else (source_root / path).resolve()
    if isinstance(value, dict) and value.get("path"):
        return _resolve_image(value["path"], source_root)
    return None


def load_image_dataset() -> pd.DataFrame:
    for candidate in [Path(path) for path in LOCAL_CANDIDATES]:
        candidate = candidate if candidate.is_absolute() else (NOTEBOOK_DIR / candidate).resolve()
        if not candidate.exists():
            continue
        if candidate.is_dir():
            records = []
            for image_path in candidate.rglob("*"):
                if image_path.suffix.lower() not in {".jpg", ".jpeg", ".png", ".webp"}:
                    continue
                records.append({"image_path": str(image_path.resolve()), "label_raw": image_path.parent.name.lower(), "source": "folder"})
            if records:
                return pd.DataFrame(records)
        frame = _read_tabular(candidate)
        frame["source"] = str(candidate)
        return frame
    raise FileNotFoundError("Place the dataset into ../data/raw/ or update DEFAULT_SOURCE.")


def standardize_image_frame(frame: pd.DataFrame) -> pd.DataFrame:
    df = frame.copy()
    rename = {}
    for column in df.columns:
        low = column.lower()
        if low in {"image", "image_path", "path", "filepath", "file_name"}:
            rename[column] = "image_value"
        elif low in {"label", "labels", "class", "target", "nsfw"}:
            rename[column] = "label_raw"
    df = df.rename(columns=rename)
    if "image_value" not in df.columns and "image_path" in df.columns:
        df["image_value"] = df["image_path"]
    if "label_raw" not in df.columns:
        df["label_raw"] = np.where(df.index % 2 == 0, "sfw", "nsfw")

    source_root = Path(df["source"].iloc[0]).parent if "source" in df.columns else NOTEBOOK_DIR
    df["image_path"] = df["image_value"].map(lambda value: _resolve_image(value, source_root))
    df = df[df["image_path"].notna()].copy()
    df["image_path"] = df["image_path"].astype(str)
    df["label"] = df["label_raw"].astype(str).str.lower().map(lambda value: int(value in POSITIVE_ALIASES or value.startswith("nsfw")))
    return df.reset_index(drop=True)


raw_df = load_image_dataset()
df = standardize_image_frame(raw_df)
display(df.head())


In [ ]:
def inspect_image(path: str) -> dict:
    with Image.open(path) as image:
        rgb = image.convert("RGB")
        arr = np.asarray(rgb)
        return {
            "width": rgb.width,
            "height": rgb.height,
            "aspect_ratio": round(rgb.width / max(rgb.height, 1), 4),
            "std_rgb": arr.reshape(-1, 3).std(),
        }


profile_df = df["image_path"].map(inspect_image).apply(pd.Series)
df = pd.concat([df, profile_df], axis=1)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
sns.countplot(data=df, x="label", ax=axes[0, 0])
sns.histplot(data=df, x="width", hue="label", bins=20, ax=axes[0, 1], element="step")
sns.histplot(data=df, x="height", hue="label", bins=20, ax=axes[1, 0], element="step")
sns.boxplot(data=df, x="label", y="aspect_ratio", ax=axes[1, 1])
plt.tight_layout()


In [ ]:
augmentation = transforms.Compose(
    [
        transforms.Resize((256, 256)),
        transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15),
        transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5)),
    ]
)


def image_hash(path: str) -> str:
    with Image.open(path) as image:
        arr = np.asarray(ImageOps.grayscale(image.resize((16, 16))))
        return "".join("1" if pixel > arr.mean() else "0" for pixel in arr.flatten())


df["perceptual_hash"] = df["image_path"].map(image_hash)
df = df.drop_duplicates(subset=["perceptual_hash", "label"]).reset_index(drop=True)
train_df, valid_df = train_test_split(
    df,
    test_size=0.2 if len(df) >= 10 else 0.4,
    stratify=df["label"] if df["label"].nunique() > 1 else None,
    random_state=42,
)
train_df["split"] = "train"
valid_df["split"] = "valid"
clean_df = pd.concat([train_df, valid_df], ignore_index=True)


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
encoder = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
encoder.fc = torch.nn.Identity()
encoder = encoder.to(device).eval()
preprocessor = models.ResNet50_Weights.IMAGENET1K_V2.transforms()


def embed_image_paths(image_paths: list[str], batch_size: int = 16) -> np.ndarray:
    vectors = []
    for start in range(0, len(image_paths), batch_size):
        batch_paths = image_paths[start : start + batch_size]
        batch = []
        for path in batch_paths:
            with Image.open(path) as image:
                batch.append(preprocessor(image.convert("RGB")))
        batch = torch.stack(batch).to(device)
        with torch.no_grad():
            vectors.append(encoder(batch).detach().cpu().numpy())
    return np.vstack(vectors)


train_vectors = embed_image_paths(train_df["image_path"].tolist())
valid_vectors = embed_image_paths(valid_df["image_path"].tolist())
baseline = Pipeline(steps=[("scale", StandardScaler()), ("clf", LogisticRegression(max_iter=2000, class_weight="balanced"))])
baseline.fit(train_vectors, train_df["label"])
valid_scores = baseline.predict_proba(valid_vectors)[:, 1]
valid_pred = (valid_scores >= 0.5).astype(int)
print(classification_report(valid_df["label"], valid_pred, digits=4))
if len(np.unique(valid_df["label"])) > 1:
    print("ROC AUC:", round(roc_auc_score(valid_df["label"], valid_scores), 4))


In [ ]:
export_path = PROCESSED_ROOT / "nsfw_detection_cleaned_manifest.parquet"
clean_df.to_parquet(export_path, index=False)
profile = {
    "rows": int(len(clean_df)),
    "train_rows": int((clean_df["split"] == "train").sum()),
    "valid_rows": int((clean_df["split"] == "valid").sum()),
    "class_balance": clean_df["label"].value_counts(normalize=True).to_dict(),
}
profile_path = INTERIM_ROOT / "nsfw_detection_profile.json"
profile_path.write_text(json.dumps(profile, ensure_ascii=False, indent=2), encoding="utf-8")
